# OpenPI pi0.5 xArm Fine-Tuning

Clean Colab pipeline for training OpenPI with a Hugging Face LeRobot dataset.

Assumptions:

- The xArm data config and `TrainConfig` entries are already manually added to `/content/openpi/src/openpi/training/config.py`.
- The manual OpenPI config uses the Hugging Face dataset repo id directly.
- Google Drive is used only for persistent caches, norm stats, checkpoints, and final exports.


## 1. Mount Drive and Configure


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess

DRIVE_ROOT = Path('/content/drive/MyDrive/embodied_ai_xarm')
OPENPI_DIR = Path('/content/openpi')
HF_LEROBOT_HOME = DRIVE_ROOT / 'lerobot'
ASSETS_BACKUP = DRIVE_ROOT / 'openpi_assets'
CHECKPOINT_BACKUP = DRIVE_ROOT / 'openpi_checkpoints'
FINAL_MODELS = DRIVE_ROOT / 'final_models'
STATE_DIR = DRIVE_ROOT / 'pipeline_state'

CONFIG_NAME = 'pi05_xarm_full_finetune'
EXP_NAME = 'pi05_xarm_full_finetune'

HF_DATASET_REPO_ID = 'YOUR_HF_USERNAME/xarm_pi05_data'  # TODO: replace this
REPO_ID = HF_DATASET_REPO_ID

FORCE_NORM_STATS = False

for path in [DRIVE_ROOT, HF_LEROBOT_HOME, ASSETS_BACKUP, CHECKPOINT_BACKUP, FINAL_MODELS, STATE_DIR]:
    path.mkdir(parents=True, exist_ok=True)
Path('/content/uv_cache').mkdir(parents=True, exist_ok=True)
Path('/content/pip_cache').mkdir(parents=True, exist_ok=True)

os.environ['HF_LEROBOT_HOME'] = str(HF_LEROBOT_HOME)
os.environ['UV_CACHE_DIR'] = '/content/uv_cache'
os.environ['PIP_CACHE_DIR'] = '/content/pip_cache'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9'
os.environ['WANDB_MODE'] = 'disabled'

def run(cmd, *, cwd=None, env=None):
    print('+', ' '.join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True)

def read_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else default

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + '\n')

def manifest_for(paths):
    h = hashlib.sha256()
    for root in map(Path, paths):
        if not root.exists():
            continue
        files = [root] if root.is_file() else sorted(p for p in root.rglob('*') if p.is_file())
        base = root.parent if root.is_file() else root
        for file in files:
            stat = file.stat()
            h.update(file.relative_to(base).as_posix().encode())
            h.update(str(stat.st_size).encode())
            h.update(str(int(stat.st_mtime)).encode())
    return h.hexdigest()

print('OpenPI dir:', OPENPI_DIR)
print('Dataset repo:', REPO_ID)
print('LeRobot cache:', HF_LEROBOT_HOME)
print('Checkpoint backup:', CHECKPOINT_BACKUP)


## 2. Check GPU


In [ ]:
!nvidia-smi


## 3. Clone and Install OpenPI


In [ ]:
%%bash
set -e
cd /content
if [ ! -d openpi ]; then
  git clone --recurse-submodules https://github.com/Physical-Intelligence/openpi.git
fi
cd /content/openpi
pip install -q uv
pip install -q -U --no-cache-dir crcmod
python -c "import crcmod._crcfunext; print('crcmod C extension ok')"
mkdir -p ~/.config/gcloud
cat > ~/.boto <<'EOF'
[GSUtil]
check_hashes = if_fast_else_skip
EOF
export UV_CACHE_DIR=/content/uv_cache
export UV_LINK_MODE=copy
GIT_LFS_SKIP_SMUDGE=1 uv sync
GIT_LFS_SKIP_SMUDGE=1 uv pip install -e .


## 4. Verify Manual OpenPI Config

Before running this cell, manually add the xArm data config and train configs to OpenPI's `src/openpi/training/config.py`. The configured `repo_id` must match `HF_DATASET_REPO_ID`. No explicit dataset download step is needed.


In [ ]:
if REPO_ID.startswith('YOUR_HF_USERNAME/'):
    raise ValueError('Set HF_DATASET_REPO_ID in Step 1 before continuing.')

openpi_config = OPENPI_DIR / 'src/openpi/training/config.py'
text = openpi_config.read_text()

assert CONFIG_NAME in text, f'Missing CONFIG_NAME={CONFIG_NAME!r} in {openpi_config}'
assert REPO_ID in text, f'Missing dataset repo id {REPO_ID!r} in {openpi_config}'

local_ckpts = OPENPI_DIR / 'checkpoints'
if local_ckpts.is_symlink() or (local_ckpts.exists() and not local_ckpts.is_dir()):
    local_ckpts.unlink()
local_ckpts.mkdir(parents=True, exist_ok=True)
CHECKPOINT_BACKUP.mkdir(parents=True, exist_ok=True)

print('Manual config verified:', openpi_config)
print('Local checkpoint dir:', local_ckpts)
!grep -n "pi05_xarm_full_finetune" /content/openpi/src/openpi/training/config.py


## 5. Compute or Restore Norm Stats

OpenPI/LeRobot will resolve the Hugging Face dataset from the `repo_id` in your manual config. Norm stats are restored from Drive when `REPO_ID` and the OpenPI config file are unchanged.


In [ ]:
config_file = OPENPI_DIR / 'src/openpi/training/config.py'
local_assets = OPENPI_DIR / 'assets' / CONFIG_NAME
backup_assets = ASSETS_BACKUP / CONFIG_NAME
norm_manifest_path = STATE_DIR / f'norm_stats_{CONFIG_NAME}.json'
manifest = {
    'config_name': CONFIG_NAME,
    'repo_id': REPO_ID,
    'config_hash': manifest_for([config_file]),
}

print('Config name:', CONFIG_NAME)
print('Expected dataset repo_id:', REPO_ID)
print('HF_LEROBOT_HOME:', os.environ.get('HF_LEROBOT_HOME'))
print('OpenPI config lines:')
!grep -n "pi05_xarm_full_finetune\|repo_id=" /content/openpi/src/openpi/training/config.py | head -40

if backup_assets.exists() and read_json(norm_manifest_path, {}) == manifest and not FORCE_NORM_STATS:
    shutil.rmtree(local_assets, ignore_errors=True)
    local_assets.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(backup_assets, local_assets)
    print('Restored norm stats:', backup_assets)
else:
    shutil.rmtree(local_assets, ignore_errors=True)
    shutil.rmtree(backup_assets, ignore_errors=True)
    run(['uv', 'run', 'python', '-u', 'scripts/compute_norm_stats.py', '--config-name', CONFIG_NAME], cwd=OPENPI_DIR)
    assert local_assets.exists(), f'Norm stats were not created: {local_assets}'
    shutil.copytree(local_assets, backup_assets)
    write_json(norm_manifest_path, manifest)
    print('Computed and backed up norm stats:', backup_assets)

!find "{local_assets}" -maxdepth 3 -type f | sort


## 6. Train From Scratch

This starts a fresh training run with `--overwrite`. Checkpoints are written under `/content/openpi/checkpoints`.


In [ ]:
!rm -rf /root/.cache/openpi/openpi-assets/checkpoints/pi05_base/params /root/.cache/openpi/openpi-assets/checkpoints/pi05_base/params.partial
!cd /content/openpi && BOTO_CONFIG=/root/.boto WANDB_PROJECT=embodied_ai_xarm WANDB_NAME={EXP_NAME} PYTHONUNBUFFERED=1 uv run python -u scripts/train.py {CONFIG_NAME} --exp-name={EXP_NAME} --overwrite


## 7. Export Latest Checkpoint for Inference


In [ ]:
import tarfile

def complete_checkpoint_steps(run_dir: Path):
    if not run_dir.exists():
        return []
    steps = []
    for path in run_dir.iterdir():
        complete = (
            path.is_dir()
            and path.name.isdigit()
            and (path / '_CHECKPOINT_METADATA').exists()
            and (path / 'assets').exists()
            and (path / 'params').exists()
        )
        if complete:
            steps.append(int(path.name))
    return sorted(steps)

local_run_dir = OPENPI_DIR / 'checkpoints' / CONFIG_NAME / EXP_NAME
drive_run_dir = CHECKPOINT_BACKUP / CONFIG_NAME / EXP_NAME
run_dir = local_run_dir if complete_checkpoint_steps(local_run_dir) else drive_run_dir
steps = complete_checkpoint_steps(run_dir)
assert steps, f'No complete checkpoints found under {local_run_dir} or {drive_run_dir}'
latest = steps[-1]

src = run_dir / str(latest)
final_root = FINAL_MODELS / f'{CONFIG_NAME}_{EXP_NAME}'
dst = final_root / str(latest)
tmp = Path('/content') / f'{CONFIG_NAME}_export'
tar_path = DRIVE_ROOT / f'{CONFIG_NAME}_{latest}_inference.tar.gz'

shutil.rmtree(tmp, ignore_errors=True)
shutil.rmtree(dst, ignore_errors=True)
tmp_step = tmp / str(latest)
tmp_step.mkdir(parents=True, exist_ok=True)
final_root.mkdir(parents=True, exist_ok=True)

run(['rsync', '-ah', '--delete', '--exclude', 'train_state', f'{src.as_posix()}/', f'{tmp_step.as_posix()}/'])
assert (tmp_step / 'assets').exists(), f'Missing assets in export: {tmp_step}'
assert (tmp_step / 'params').exists(), f'Missing params in export: {tmp_step}'
run(['rsync', '-ah', '--delete', f'{tmp_step.as_posix()}/', f'{dst.as_posix()}/'])

with tarfile.open(tar_path, 'w:gz') as tar:
    tar.add(tmp_step, arcname=str(latest))

readme = (
    f"OpenPI xArm pi0.5 inference export.\n\n"
    f"Config name: {CONFIG_NAME}\n"
    f"Experiment name: {EXP_NAME}\n"
    f"Checkpoint step: {latest}\n\n"
    f"Serve with:\n"
    f"uv run scripts/serve_policy.py policy:checkpoint \\\n"
    f"  --policy.config={CONFIG_NAME} \\\n"
    f"  --policy.dir={dst}\n"
)
(final_root / 'README.txt').write_text(readme)

print('Exported from:', src)
print('Exported directory:', dst)
print('Exported archive:', tar_path)
!ls -lh "{tar_path}"
